# 02 — Triage Modelling

Creates the model-ready feature frame from the cleaned `sample/joined` table, then trains and compares the calibrated logistic baseline and XGBoost challenger on a temporal holdout.

In [1]:
import io
import os
import sys

import boto3
import pandas as pd
import plotly.express as px
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

from sift.config import load_settings
from sift.features.build_features import add_domain_features, add_request_features, make_label
from sift.models.triage import train_baseline, train_challenger

settings = load_settings()
BUCKET = settings.s3_bucket or 'sift-piracy-data'
ACTION_THRESHOLD = settings.get('label', 'action_threshold', default=0.5)
CUTOFF_DATE = settings.get('split', 'cutoff_date', default='2024-09-01')
print(f'bucket={BUCKET}  action_threshold={ACTION_THRESHOLD}  cutoff={CUTOFF_DATE}')

bucket=sift-piracy-data  action_threshold=0.5  cutoff=2024-09-01


In [2]:
def read_parquet_prefix(bucket: str, prefix: str, max_parts: int = 999) -> pd.DataFrame:
    """Download Parquet parts from an S3 prefix and concatenate."""
    s3 = boto3.client('s3')
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    keys = sorted(
        o['Key'] for o in resp.get('Contents', []) if o['Key'].endswith('.parquet')
    )[:max_parts]
    frames = []
    for key in keys:
        obj = s3.get_object(Bucket=bucket, Key=key)
        frames.append(pd.read_parquet(io.BytesIO(obj['Body'].read())))
    return pd.concat(frames, ignore_index=True)


df = read_parquet_prefix(BUCKET, 'sample/joined/')
df['date'] = pd.to_datetime(df['date'])
print(f'joined sample: {len(df):,} rows')
print(f'date range: {df.date.min().date()} -> {df.date.max().date()}')

joined sample: 3,661,175 rows
date range: 2011-05-20 -> 2026-05-04


In [5]:
df

,request_id,domain,urls_specified,urls_removed,urls_no_action,urls_not_in_index,urls_pending,removal_rate,from_abuser,date,reporting_org_id,reporting_org,copyright_owner_id,copyright_owner,request_urls_specified,request_urls_removed,request_removal_rate,request_from_abuser
0,10054577,123unblock.surf,1352,1352,0,0,0,1.0,False,2020-10-26,141552,MG Premium Ltd.,70158,MG Premium Ltd,13918,10831,0.778201,False
1,10054577,1337x.to,2,2,0,0,0,1.0,False,2020-10-26,141552,MG Premium Ltd.,70158,MG Premium Ltd,13918,10831,0.778201,False
2,10054577,ddownload.com,2,0,0,2,0,0.0,False,2020-10-26,141552,MG Premium Ltd.,70158,MG Premium Ltd,13918,10831,0.778201,False
3,10054577,dood.to,2,0,0,2,0,0.0,False,2020-10-26,141552,MG Premium Ltd.,70158,MG Premium Ltd,13918,10831,0.778201,False
4,10054577,fastfile.cc,2,0,0,2,0,0.0,False,2020-10-26,141552,MG Premium Ltd.,70158,MG Premium Ltd,13918,10831,0.778201,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3661170,9397946,limetorrents.info,1,1,0,0,0,1.0,False,2020-06-16,121710,proMedia,200788,Rh Music,2,1,0.500000,False
3661171,9726294,herrutor.site,1,1,0,0,0,1.0,False,2020-08-27,206416,Group-IB,305064,ООО «ИBИ.РУ»,5,4,0.800000,False
3661172,9726294,kinomops.su,1,1,0,0,0,1.0,False,2020-08-27,206416,Group-IB,305064,ООО «ИBИ.РУ»,5,4,0.800000,False
3661173,9726294,limetorrents.info,1,1,0,0,0,1.0,False,2020-08-27,206416,Group-IB,305064,ООО «ИBИ.РУ»,5,4,0.800000,False


In [4]:
model_df = df.copy()
model_df = add_request_features(model_df, input_prefix='request_', output_prefix='request_')
model_df = add_domain_features(model_df, input_prefix='domain_')
model_df = make_label(model_df, threshold=ACTION_THRESHOLD, target_col='domain_removal_rate')

FEATURE_COLS = [
    'request_urls_log1p',
    'request_not_in_index_ratio',
    'request_no_action_ratio',
    'request_dow',
    'request_month',
    'request_org_historical_hit_rate',
    'request_owner_historical_hit_rate',
    'domain_has_suspicious_token',
    'suspicious_token_count',
    'tld_is_risky',
    'domain_length',
    'digit_ratio',
    'hyphen_count',
    'domain_historical_hit_rate',
]

print(f'feature columns: {len(FEATURE_COLS)}')
model_df[FEATURE_COLS + ['action']].head()

feature columns: 14


,urls_log1p,not_in_index_ratio,no_action_ratio,dow,month,org_historical_hit_rate,owner_historical_hit_rate,domain_has_suspicious_token,suspicious_token_count,tld_is_risky,domain_length,digit_ratio,hyphen_count,domain_historical_hit_rate,action
0,7.210080,0.0,0.0,0,10,0.505814,0.771841,0,0,0,15,0.2,0,0.750000,1
1,1.098612,0.0,0.0,0,10,0.505814,0.771841,0,0,0,8,0.5,0,0.751119,1
2,1.098612,1.0,0.0,0,10,0.505814,0.771841,0,0,0,13,0.0,0,0.229170,0
3,1.098612,1.0,0.0,0,10,0.505814,0.771841,0,0,0,7,0.0,0,1.000000,0
4,1.098612,1.0,0.0,0,10,0.505814,0.771841,0,0,1,11,0.0,0,0.714286,0


In [ ]:
cutoff = pd.Timestamp(CUTOFF_DATE)
train = model_df[model_df['date'] < cutoff].copy()
holdout = model_df[model_df['date'] >= cutoff].copy()

X_train = train[FEATURE_COLS]
y_train = train['action']
X_holdout = holdout[FEATURE_COLS]
y_holdout = holdout['action']

print(f'train:   {len(train):,} rows  positive={y_train.mean():.3f}')
print(f'holdout: {len(holdout):,} rows  positive={y_holdout.mean():.3f}')

In [ ]:
def evaluate_model(name, model):
    scores = model.predict_proba(X_holdout)
    precision, recall, _ = precision_recall_curve(y_holdout, scores)
    frac_pos, mean_pred = calibration_curve(y_holdout, scores, n_bins=10, strategy='quantile')
    return {
        'name': name,
        'model': model,
        'scores': scores,
        'roc_auc': roc_auc_score(y_holdout, scores),
        'average_precision': average_precision_score(y_holdout, scores),
        'pr_curve': pd.DataFrame({'precision': precision, 'recall': recall}),
        'calibration': pd.DataFrame({'mean_predicted': mean_pred, 'fraction_positive': frac_pos}),
    }


results = []

baseline = train_baseline(X_train, y_train)
results.append(evaluate_model('Logistic regression', baseline))

try:
    challenger = train_challenger(X_train, y_train)
    results.append(evaluate_model('XGBoost', challenger))
except RuntimeError as exc:
    print(f'XGBoost challenger skipped locally: {exc}')

metrics = pd.DataFrame(
    [{'model': r['name'], 'roc_auc': r['roc_auc'], 'average_precision': r['average_precision']} for r in results]
).sort_values('average_precision', ascending=False)
metrics

In [ ]:
pr_frames = []
calibration_frames = []
for result in results:
    pr = result['pr_curve'].copy()
    pr['model'] = result['name']
    pr_frames.append(pr)

    cal = result['calibration'].copy()
    cal['model'] = result['name']
    calibration_frames.append(cal)

fig = px.line(
    pd.concat(pr_frames),
    x='recall',
    y='precision',
    color='model',
    title='Precision-recall on temporal holdout',
)
fig.show()

fig = px.line(
    pd.concat(calibration_frames),
    x='mean_predicted',
    y='fraction_positive',
    color='model',
    markers=True,
    title='Calibration curve on temporal holdout',
)
fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=1, line={'dash': 'dash', 'color': 'grey'})
fig.show()

## Model choice

Pick the model with the better ranking behaviour on the temporal holdout. For this triage workflow, average precision / precision-recall behaviour matters more than raw accuracy because analysts work from the top of the ranked list. Record the chosen model and the evidence here after running the notebook.